In [1]:
class CandidateElimination:

    def __init__(self, attributes):
        self.attributes = attributes
        self.S = []                    # Most specific hypotheses
        self.G = [['?'] * len(attributes)]  # Most general hypotheses

    def process_example(self, instance, label):

        print(f"Processing: {instance} -> {label}")

        if label == 'Yes':

            # Remove from G any inconsistent hypotheses
            self.G = [
                g for g in self.G
                if self.consistent(instance, g)
            ]

            if not self.S:
                # Initialize S with this positive example
                self.S = [instance.copy()]

            else:
                # Generalize S
                new_S = []

                for s in self.S:

                    if self.consistent(instance, s):
                        new_S.append(s)

                    else:
                        # Find minimal generalizations
                        generalized = s.copy()

                        for i in range(len(s)):
                            if s[i] != instance[i]:
                                generalized[i] = '?'

                        # Only keep if it is more specific
                        # than some hypothesis in G
                        if any(
                            self.more_general(g, generalized)
                            for g in self.G
                        ):
                            new_S.append(generalized)

                # Update S and remove duplicates
                self.S = []

                for s in new_S:
                    if s not in self.S:
                        self.S.append(s)

                # Remove more specific hypotheses from S
                self.S = [
                    s for s in self.S
                    if not any(
                        s != s2 and
                        self.more_specific(s, s2)
                        for s2 in self.S
                    )
                ]

        else:
            # Negative example

            # Remove from S any inconsistent hypotheses
            self.S = [
                s for s in self.S
                if not self.consistent(instance, s)
            ]

            # Specialize G
            new_G = []

            for g in self.G:

                if not self.consistent(instance, g):
                    new_G.append(g)

                else:
                    # Specialize the general hypothesis
                    for i in range(len(g)):

                        if g[i] == '?':

                            for value in self.attributes[i]:

                                if value != instance[i]:

                                    specialized = g.copy()
                                    specialized[i] = value

                                    # Keep if more general than
                                    # some hypothesis in S
                                    if (
                                        any(
                                            self.more_general(
                                                specialized, s
                                            )
                                            for s in self.S
                                        )
                                        or not self.S
                                    ):
                                        if specialized not in new_G:
                                            new_G.append(specialized)

            # Update G and remove duplicates
            self.G = []

            for g in new_G:
                if g not in self.G:
                    self.G.append(g)

            # Remove less general hypotheses from G
            self.G = [
                g for g in self.G
                if not any(
                    g != g2 and
                    self.more_general(g2, g)
                    for g2 in self.G
                )
            ]

    def consistent(self, instance, hypothesis):

        for attr, hyp_val in zip(instance, hypothesis):

            if hyp_val != '?' and hyp_val != attr:
                return False

        return True

    def more_general(self, h1, h2):

        for a1, a2 in zip(h1, h2):

            if a1 != '?' and a2 == '?':
                return False

            if (
                a1 != '?' and
                a2 != '?' and
                a1 != a2
            ):
                return False

        return True

    def more_specific(self, h1, h2):
        return self.more_general(h2, h1)

    def print_hypotheses(self):

        print("Current Version Space:")

        if not self.S:
            print(" S: EMPTY (no specific hypothesis)")
        else:
            for i, s in enumerate(self.S):
                print(f" S{i+1}: <{','.join(s)}>")

        if not self.G:
            print(" G: EMPTY (no general hypothesis)")
        else:
            for i, g in enumerate(self.G):
                print(f" G{i+1}: <{','.join(g)}>")

        print()


def main():

    # Define the EnjoySport dataset
    data = [
        ['Sunny', 'Warm', 'Normal', 'Strong', 'Yes'],
        ['Sunny', 'Warm', 'High', 'Strong', 'Yes'],
        ['Rainy', 'Cold', 'High', 'Strong', 'No'],
        ['Sunny', 'Warm', 'High', 'Weak', 'Yes']
    ]

    # Define attribute domains
    attributes = [
        ['Sunny', 'Rainy'],     # Sky
        ['Warm', 'Cold'],       # AirTemp
        ['Normal', 'High'],      # Humidity
        ['Strong', 'Weak']      # Wind
    ]

    # Column names
    column_names = [
        'Sky',
        'AirTemp',
        'Humidity',
        'Wind',
        'EnjoySport'
    ]

    print("EnjoySport Dataset:")

    for i, example in enumerate(data):
        print(f"Example {i+1}: {example}")

    print()

    # Initialize Candidate Elimination algorithm
    ce = CandidateElimination(attributes)

    print("Initial State:")
    ce.print_hypotheses()

    # Process each example
    for i, example in enumerate(data):

        instance = example[:-1]
        label = example[-1]

        print(f"Step {i+1}:")

        ce.process_example(instance, label)

        ce.print_hypotheses()

    print("Final Version Space:")
    ce.print_hypotheses()

    # Interpretation of results
    print("Analysis:")

    if (
        len(ce.S) == 1 and
        len(ce.G) == 1 and
        ce.S[0] == ce.G[0]
    ):

        print(
            "The algorithm converged to a "
            "single consistent hypothesis:"
        )

        print(
            f" Hypothesis: <{','.join(ce.S[0])}>"
        )

    elif len(ce.S) > 0 and len(ce.G) > 0:

        print(
            "The version space contains "
            "multiple hypotheses."
        )

        print(
            f"Most specific: <{','.join(ce.S[0])}>"
        )

        print(
            f"Most general: <{','.join(ce.G[0])}>"
        )

        print(
            f"Total hypotheses in version space: "
            f"{len(ce.S) + len(ce.G)}"
        )

    else:
        print(
            "No consistent hypothesis found "
            "for the training data."
        )


if __name__ == "__main__":
    main()

EnjoySport Dataset:
Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Yes']
Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Yes']
Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'No']
Example 4: ['Sunny', 'Warm', 'High', 'Weak', 'Yes']

Initial State:
Current Version Space:
 S: EMPTY (no specific hypothesis)
 G1: <?,?,?,?>

Step 1:
Processing: ['Sunny', 'Warm', 'Normal', 'Strong'] -> Yes
Current Version Space:
 S1: <Sunny,Warm,Normal,Strong>
 G1: <?,?,?,?>

Step 2:
Processing: ['Sunny', 'Warm', 'High', 'Strong'] -> Yes
Current Version Space:
 S1: <Sunny,Warm,?,Strong>
 G1: <?,?,?,?>

Step 3:
Processing: ['Rainy', 'Cold', 'High', 'Strong'] -> No
Current Version Space:
 S1: <Sunny,Warm,?,Strong>
 G1: <Sunny,?,?,?>
 G2: <?,Warm,?,?>

Step 4:
Processing: ['Sunny', 'Warm', 'High', 'Weak'] -> Yes
Current Version Space:
 S1: <Sunny,Warm,?,?>
 G1: <Sunny,?,?,?>
 G2: <?,Warm,?,?>

Final Version Space:
Current Version Space:
 S1: <Sunny,Warm,?,?>
 G1: <Sunny,?,?,?>
 G2: <?,Warm,?,?>

Analysis:
Th